
# GVH Diagonal Cubic 0.3.2.3 — Explicit Field Equations and Tensor Audit

**Partie C — weak-field theoretical closure**  
**Auteur :** Charlemagne O Laurince  
**Version :** 0.3.2.3  
**Statut :** tensor/equation audit of the 0.3.2.2 candidate action  
**Important :** the timelike/vector action remains a **candidate ansatz**, not an established GVH law.

---

## Objective

Notebook 0.3.2.2 introduced the candidate action

\[
S_{\rm cand}
=
\frac{1}{16\pi G}
\int d^4x\,\sqrt{-g}
\left[
R
-
K^{\alpha\beta}{}_{\mu\nu}
\nabla_\alpha u^\mu\nabla_\beta u^\nu
+
\lambda(u^\mu u_\mu+1)
\right]
+
S_m.
\]

The present notebook audits the corresponding field-equation structure before any static weak-field reduction.

We require three equations:

\[
\frac{\delta S}{\delta \lambda}=0,
\qquad
\frac{\delta S}{\delta u^\mu}=0,
\qquad
\frac{\delta S}{\delta g^{\mu\nu}}=0.
\]

The central question is not merely whether equations can be written, but whether they are sufficiently explicit and internally constrained to justify the next step toward \(a_1,a_2,b_1\).



## 0. Epistemic classification

Every object is assigned one of the following statuses:

- `ESTABLISHED_REFERENCE` — standard variational/geometric structure used as reference;
- `CANDIDATE_DERIVED` — derived conditionally from the 0.3.2.2 candidate action;
- `FORM_TO_VERIFY` — explicit tensor form proposed for audit but not yet promoted;
- `GVH_DERIVED` — result already inherited from the validated GVH chain;
- `BLOCKED` — derivation not completed;
- `NOT_A_PREDICTION` — cannot yet be used as an observational GVH prediction.


In [1]:

from __future__ import annotations

from dataclasses import dataclass, asdict
from enum import Enum
from pathlib import Path
import json
import platform
import sys

import numpy as np
import pandas as pd
import sympy as sp

NOTEBOOK_ID = "GVH_Diagonal_Cubic_0.3.2.3"
VERSION = "0.3.2.3"

class Status(str, Enum):
    ESTABLISHED_REFERENCE = "ESTABLISHED_REFERENCE"
    CANDIDATE_DERIVED = "CANDIDATE_DERIVED"
    FORM_TO_VERIFY = "FORM_TO_VERIFY"
    GVH_DERIVED = "GVH_DERIVED"
    BLOCKED = "BLOCKED"
    NOT_A_PREDICTION = "NOT_A_PREDICTION"

print(NOTEBOOK_ID, VERSION)
print("Python:", sys.version.split()[0])
print("SymPy:", sp.__version__)


GVH_Diagonal_Cubic_0.3.2.3 0.3.2.3
Python: 3.12.13
SymPy: 1.14.0



## 1. Candidate tensor \(K^{\alpha\beta}{}_{\mu\nu}\)

We preserve the 0.3.2.2 convention

\[
K^{\alpha\beta}{}_{\mu\nu}
=
c_1 g^{\alpha\beta}g_{\mu\nu}
+
c_2 \delta^\alpha_{\mu}\delta^\beta_{\nu}
+
c_3 \delta^\alpha_{\nu}\delta^\beta_{\mu}
-
c_4 u^\alpha u^\beta g_{\mu\nu}.
\]

Define

\[
J^\alpha{}_\mu
=
K^{\alpha\beta}{}_{\mu\nu}\nabla_\beta u^\nu,
\qquad
a^\mu=u^\nu\nabla_\nu u^\mu.
\]

The \(c_i\) remain free candidate couplings.


In [2]:

c1, c2, c3, c4, G = sp.symbols("c1 c2 c3 c4 G", real=True)

candidate_objects = pd.DataFrame([
    {"object":"K^{ab}_{mn}", "status":"CANDIDATE_DERIVED", "source":"0.3.2.2 ansatz"},
    {"object":"J^a_m", "status":"CANDIDATE_DERIVED", "source":"definition from K and nabla u"},
    {"object":"a^m", "status":"ESTABLISHED_REFERENCE", "source":"vector acceleration definition"},
    {"object":"c1..c4", "status":"FORM_TO_VERIFY", "source":"free candidate couplings"},
])
candidate_objects


,object,status,source
0,K^{ab}_{mn},CANDIDATE_DERIVED,0.3.2.2 ansatz
1,J^a_m,CANDIDATE_DERIVED,definition from K and nabla u
2,a^m,ESTABLISHED_REFERENCE,vector acceleration definition
3,c1..c4,FORM_TO_VERIFY,free candidate couplings



## 2. Multiplier equation

Variation with respect to \(\lambda\) gives

\[
\boxed{u^\mu u_\mu=-1}.
\]

This equation is exact **conditional on the candidate action**.


In [3]:

u2 = sp.symbols("u_squared", real=True)
constraint_eq = sp.Eq(u2 + 1, 0)
constraint_solution = sp.solve(constraint_eq, u2)

assert constraint_solution == [-1]
print("PASS — unit timelike constraint:", constraint_solution)


PASS — unit timelike constraint: [-1]



## 3. Vector equation — explicit candidate structure

For the chosen quadratic first-derivative ansatz, record the vector equation in the form

\[
\boxed{
\nabla_\alpha J^\alpha{}_\mu
+
c_4 a_\alpha\nabla_\mu u^\alpha
+
\lambda u_\mu
=
0
}
\]

subject to the sign/index convention of 0.3.2.2.

At this stage the equation is labeled `FORM_TO_VERIFY`: the structure is explicit enough for downstream tensor checks, but the notebook will not pretend that a full component-by-component variation has already been independently reproduced by a general-purpose tensor CAS.


In [4]:

vector_eom_terms = pd.DataFrame([
    {"term":"nabla_alpha J^alpha_mu", "role":"kinetic Euler-Lagrange divergence", "status":"FORM_TO_VERIFY"},
    {"term":"c4 a_alpha nabla_mu u^alpha", "role":"acceleration-dependent contribution", "status":"FORM_TO_VERIFY"},
    {"term":"lambda u_mu", "role":"constraint reaction", "status":"CANDIDATE_DERIVED"},
])

vector_eom_terms


,term,role,status
0,nabla_alpha J^alpha_mu,kinetic Euler-Lagrange divergence,FORM_TO_VERIFY
1,c4 a_alpha nabla_mu u^alpha,acceleration-dependent contribution,FORM_TO_VERIFY
2,lambda u_mu,constraint reaction,CANDIDATE_DERIVED



## 4. Solving for the multiplier by contraction

Contract the candidate vector equation with \(u^\mu\):

\[
u^\mu\nabla_\alpha J^\alpha{}_\mu
+
c_4 u^\mu a_\alpha\nabla_\mu u^\alpha
+
\lambda\,u^\mu u_\mu
=
0.
\]

Using \(u^\mu u_\mu=-1\), one obtains formally

\[
\boxed{
\lambda
=
u^\mu\nabla_\alpha J^\alpha{}_\mu
+
c_4 u^\mu a_\alpha\nabla_\mu u^\alpha
}.
\]

This removes \(\lambda\) as an independent physical coupling once the vector configuration is known.


In [5]:

lambda_symbol = sp.symbols("lambda", real=True)

multiplier_audit = {
    "constraint_used": "u^mu u_mu = -1",
    "lambda_is_independent_coupling": False,
    "lambda_role": "constraint reaction determined by vector equation + configuration",
    "status": "CANDIDATE_DERIVED",
}
multiplier_audit


{'constraint_used': 'u^mu u_mu = -1',
 'lambda_is_independent_coupling': False,
 'lambda_role': 'constraint reaction determined by vector equation + configuration',
 'status': 'CANDIDATE_DERIVED'}


## 5. Metric equation — variational definition

The metric equation is

\[
\boxed{
G_{\mu\nu}
=
8\pi G\,T_{\mu\nu}^{(m)}
+
T_{\mu\nu}^{(u)}
}
\]

with

\[
\boxed{
T_{\mu\nu}^{(u)}
=
-\frac{2}{\sqrt{-g}}
\frac{\delta S_u}{\delta g^{\mu\nu}}
}.
\]

This definition is exact conditional on the candidate action.

The difficult part is the explicit expansion of \(T_{\mu\nu}^{(u)}\), because the metric enters through:

- \(\sqrt{-g}\),
- index raising/lowering,
- \(K^{\alpha\beta}{}_{\mu\nu}\),
- covariant derivatives,
- the acceleration term,
- the normalization constraint.


In [6]:

metric_variation_sources = pd.DataFrame([
    {"source":"sqrt(-g)", "must_vary":True},
    {"source":"metric contractions", "must_vary":True},
    {"source":"K tensor", "must_vary":True},
    {"source":"connection inside nabla u", "must_vary":True},
    {"source":"acceleration a^mu", "must_vary":True},
    {"source":"constraint u^mu u_mu", "must_vary":True},
])
metric_variation_sources


,source,must_vary
0,sqrt(-g),True
1,metric contractions,True
2,K tensor,True
3,connection inside nabla u,True
4,acceleration a^mu,True
5,constraint u^mu u_mu,True



## 6. Explicit candidate stress-tensor decomposition

For audit purposes, decompose

\[
T_{\mu\nu}^{(u)}
=
T_{\mu\nu}^{(J)}
+
T_{\mu\nu}^{(c_1)}
+
T_{\mu\nu}^{(c_4)}
+
T_{\mu\nu}^{(\lambda)}
+
T_{\mu\nu}^{(L)}.
\]

A useful explicit **form-to-verify** is

\[
T_{\mu\nu}^{(u)}
=
\nabla_\alpha
\left(
J^\alpha{}_{(\mu}u_{\nu)}
+
J_{(\mu}{}^\alpha u_{\nu)}
-
J_{(\mu\nu)}u^\alpha
\right)
+
c_1
\left[
(\nabla_\mu u_\alpha)(\nabla_\nu u^\alpha)
-
(\nabla_\alpha u_\mu)(\nabla^\alpha u_\nu)
\right]
+
c_4 a_\mu a_\nu
+
\lambda u_\mu u_\nu
-
\frac12 g_{\mu\nu}\mathcal L_u .
\]

**Important:** this expression is stored as `FORM_TO_VERIFY`, not as a newly proven GVH tensor identity. The next component checks test necessary properties but do not replace a full independent symbolic variation.


In [7]:

stress_decomposition = pd.DataFrame([
    {"piece":"divergence/J piece", "status":"FORM_TO_VERIFY"},
    {"piece":"c1 derivative bilinear", "status":"FORM_TO_VERIFY"},
    {"piece":"c4 acceleration square", "status":"FORM_TO_VERIFY"},
    {"piece":"lambda u_mu u_nu", "status":"CANDIDATE_DERIVED"},
    {"piece":"-1/2 g_mu_nu L_u", "status":"FORM_TO_VERIFY"},
])

stress_decomposition


,piece,status
0,divergence/J piece,FORM_TO_VERIFY
1,c1 derivative bilinear,FORM_TO_VERIFY
2,c4 acceleration square,FORM_TO_VERIFY
3,lambda u_mu u_nu,CANDIDATE_DERIVED
4,-1/2 g_mu_nu L_u,FORM_TO_VERIFY



## 7. Symmetry audit

Because \(T_{\mu\nu}^{(u)}\) is defined by variation with respect to the symmetric metric \(g^{\mu\nu}\), the final tensor must satisfy

\[
T_{\mu\nu}^{(u)}=T_{\nu\mu}^{(u)}.
\]

We inspect each displayed contribution at the structural level.


In [8]:

symmetry_audit = pd.DataFrame([
    {"piece":"J divergence with explicit (mu nu) symmetrization", "symmetric":True},
    {"piece":"c1 bilinear first term", "symmetric_under_mu_nu":True},
    {"piece":"c1 bilinear second term", "symmetric_under_mu_nu":True},
    {"piece":"c4 a_mu a_nu", "symmetric":True},
    {"piece":"lambda u_mu u_nu", "symmetric":True},
    {"piece":"g_mu_nu L_u", "symmetric":True},
])

assert symmetry_audit.filter(like="symmetric").fillna(True).all(axis=None)
print("PASS — displayed stress-tensor pieces are structurally symmetric.")
symmetry_audit


PASS — displayed stress-tensor pieces are structurally symmetric.


/tmp/ipykernel_964/669856673.py:10: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  assert symmetry_audit.filter(like="symmetric").fillna(True).all(axis=None)


,piece,symmetric,symmetric_under_mu_nu
0,J divergence with explicit (mu nu) symmetrization,True,NaN
1,c1 bilinear first term,NaN,True
2,c1 bilinear second term,NaN,True
3,c4 a_mu a_nu,True,NaN
4,lambda u_mu u_nu,True,NaN
5,g_mu_nu L_u,True,NaN



## 8. Flat-background sanity check

Take

\[
g_{\mu\nu}=\eta_{\mu\nu},
\qquad
u^\mu=(1,0,0,0)
\]

with constant components.

Then

\[
\nabla_\alpha u^\mu=0,\qquad
a^\mu=0,\qquad
J^\alpha{}_\mu=0.
\]

The vector equation reduces to

\[
\lambda u_\mu=0
\quad\Rightarrow\quad
\lambda=0,
\]

and the candidate vector stress tensor vanishes.

Thus Minkowski + constant unit timelike vector is a basic vacuum control solution of the candidate system.


In [9]:

flat_control = {
    "nabla_u_zero": True,
    "a_zero": True,
    "J_zero": True,
    "lambda_zero_from_vector_EOM": True,
    "T_u_zero": True,
    "candidate_vacuum_control_pass": True,
}

assert all(flat_control.values())
flat_control


{'nabla_u_zero': True,
 'a_zero': True,
 'J_zero': True,
 'lambda_zero_from_vector_EOM': True,
 'T_u_zero': True,
 'candidate_vacuum_control_pass': True}


## 9. Decoupling / GR-recovery audit

In the control limit

\[
c_1,c_2,c_3,c_4\rightarrow0
\]

and with no surviving independent vector stress,

\[
T_{\mu\nu}^{(u)}\rightarrow0.
\]

The metric equation reduces to

\[
G_{\mu\nu}=8\pi G\,T_{\mu\nu}^{(m)}.
\]

This establishes a **structural GR-recovery route**, not global equivalence to GR for arbitrary candidate-vector boundary conditions.


In [10]:

gr_recovery = {
    "all_ci_zero": {str(c1):0, str(c2):0, str(c3):0, str(c4):0},
    "vector_stress_expected_zero_in_control_sector": True,
    "metric_equation_reduces_to_GR_structure": True,
    "global_equivalence_proved": False,
}
gr_recovery


{'all_ci_zero': {'c1': 0, 'c2': 0, 'c3': 0, 'c4': 0},
 'vector_stress_expected_zero_in_control_sector': True,
 'metric_equation_reduces_to_GR_structure': True,
 'global_equivalence_proved': False}


## 10. Bianchi / conservation consistency gate

The Einstein tensor obeys

\[
\nabla^\mu G_{\mu\nu}=0.
\]

Therefore the candidate metric equation requires

\[
\nabla^\mu
\left(
8\pi G\,T_{\mu\nu}^{(m)}
+
T_{\mu\nu}^{(u)}
\right)
=0.
\]

If matter is separately covariantly conserved,

\[
\nabla^\mu T_{\mu\nu}^{(m)}=0,
\]

then on-shell consistency requires

\[
\nabla^\mu T_{\mu\nu}^{(u)}=0
\]

when the vector equation and normalization constraint hold.

This notebook records this as an **on-shell consistency requirement**. It does not claim to have algebraically proven the full divergence cancellation in arbitrary coordinates.


In [11]:

conservation_gate = pd.DataFrame([
    {"condition":"Bianchi identity", "status":"ESTABLISHED_REFERENCE", "pass":True},
    {"condition":"matter separate conservation", "status":"ASSUMPTION_FOR_GATE", "pass":True},
    {"condition":"vector EOM available", "status":"FORM_TO_VERIFY", "pass":True},
    {"condition":"constraint available", "status":"CANDIDATE_DERIVED", "pass":True},
    {"condition":"full arbitrary-coordinate divergence cancellation explicitly proven", "status":"BLOCKED", "pass":False},
])
conservation_gate


,condition,status,pass
0,Bianchi identity,ESTABLISHED_REFERENCE,True
1,matter separate conservation,ASSUMPTION_FOR_GATE,True
2,vector EOM available,FORM_TO_VERIFY,True
3,constraint available,CANDIDATE_DERIVED,True
4,full arbitrary-coordinate divergence cancellat...,BLOCKED,False



## 11. Coupling-count audit after explicit EOM structure

The field-equation structure does not automatically determine

\[
c_1,c_2,c_3,c_4.
\]

Thus even if the tensor equations are correct, the theory can remain a four-parameter family.

The key distinction is:

\[
\text{field equations exist}
\;\not\Rightarrow\;
\text{unique physical theory}.
\]


In [12]:

parameter_audit = {
    "candidate_couplings": 4,
    "couplings_fixed_by_tensor_structure_alone": 0,
    "relations_among_ci_derived_here": 0,
    "unique_GVH_action_selected": False,
    "parameter_family_remains": True,
}
parameter_audit


{'candidate_couplings': 4,
 'couplings_fixed_by_tensor_structure_alone': 0,
 'relations_among_ci_derived_here': 0,
 'unique_GVH_action_selected': False,
 'parameter_family_remains': True}


## 12. Readiness for static weak-field reduction

Before 0.3.2.4 is allowed to solve the static spherical sector, we require:

1. normalization equation explicit;
2. vector EOM explicit;
3. metric equation explicit;
4. stress tensor structurally symmetric;
5. basic flat/GR controls pass;
6. unresolved tensor identities clearly marked;
7. no claim that \(c_i\) are already determined.

The weak-field reduction may proceed as a **candidate-family calculation** even if the \(c_i\) remain free, provided the result is not mislabeled as a unique GVH prediction.


In [13]:

weak_field_readiness = {
    "constraint_explicit": True,
    "vector_EOM_explicit_form": True,
    "metric_EOM_explicit_definition": True,
    "stress_tensor_explicit_form_to_verify": True,
    "stress_symmetry_structural_pass": True,
    "flat_control_pass": True,
    "GR_recovery_route_pass": True,
    "full_tensor_variation_independently_verified": False,
    "ci_uniquely_fixed": False,
}

candidate_family_ready = all([
    weak_field_readiness["constraint_explicit"],
    weak_field_readiness["vector_EOM_explicit_form"],
    weak_field_readiness["metric_EOM_explicit_definition"],
    weak_field_readiness["stress_tensor_explicit_form_to_verify"],
    weak_field_readiness["stress_symmetry_structural_pass"],
    weak_field_readiness["flat_control_pass"],
    weak_field_readiness["GR_recovery_route_pass"],
])

unique_GVH_prediction_ready = (
    candidate_family_ready
    and weak_field_readiness["full_tensor_variation_independently_verified"]
    and weak_field_readiness["ci_uniquely_fixed"]
)

candidate_family_ready, unique_GVH_prediction_ready


(True, False)


## 13. Final status logic

Expected outcomes:

- `PASS-EXPLICIT-EOM-CANDIDATE-FAMILY_READY-FOR-WEAK-FIELD`
- `PASS-UNIQUE-GVH-EOM_READY` only if the tensor variation is independently verified **and** GVH fixes the couplings;
- `FAIL-TENSOR-STRUCTURE` if basic consistency checks fail.

At the current stage, only the first status is scientifically justified.


In [14]:

if not candidate_family_ready:
    FINAL_STATUS = "FAIL-TENSOR-STRUCTURE"
elif unique_GVH_prediction_ready:
    FINAL_STATUS = "PASS-UNIQUE-GVH-EOM_READY"
else:
    FINAL_STATUS = "PASS-EXPLICIT-EOM-CANDIDATE-FAMILY_READY-FOR-WEAK-FIELD_BLOCKED-UNIQUE-PREDICTION"

print("FINAL STATUS:", FINAL_STATUS)


FINAL STATUS: PASS-EXPLICIT-EOM-CANDIDATE-FAMILY_READY-FOR-WEAK-FIELD_BLOCKED-UNIQUE-PREDICTION



## 14. Internal audit


In [15]:

tests = {
    "unit_constraint": constraint_solution == [-1],
    "stress_structural_symmetry": bool(symmetry_audit.filter(like="symmetric").fillna(True).all(axis=None)),
    "flat_control": all(flat_control.values()),
    "candidate_family_ready": candidate_family_ready,
    "unique_prediction_still_blocked": not unique_GVH_prediction_ready,
    "four_free_candidate_couplings_preserved": parameter_audit["candidate_couplings"] == 4,
}

for key, value in tests.items():
    print(f"{key}: {value}")

assert all(tests.values())


unit_constraint: True
stress_structural_symmetry: True
flat_control: True
candidate_family_ready: True
unique_prediction_still_blocked: True
four_free_candidate_couplings_preserved: True


/tmp/ipykernel_964/4060817905.py:3: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  "stress_structural_symmetry": bool(symmetry_audit.filter(like="symmetric").fillna(True).all(axis=None)),



## 15. Machine-readable artifact

This artifact records the **candidate-family tensor status**, not an observational result.


In [16]:

artifact = {
    "notebook": NOTEBOOK_ID,
    "version": VERSION,
    "final_status": FINAL_STATUS,
    "action_status": "CANDIDATE_ANSATZ",
    "constraint_equation": "u^mu u_mu = -1",
    "vector_equation_status": "EXPLICIT_FORM_TO_VERIFY",
    "metric_equation_status": "EXPLICIT_VARIATIONAL_DEFINITION",
    "vector_stress_tensor_status": "EXPLICIT_FORM_TO_VERIFY_NOT_INDEPENDENTLY_CAS_VERIFIED",
    "stress_tensor_structural_symmetry": True,
    "flat_background_control": True,
    "GR_recovery_route": True,
    "full_on_shell_divergence_identity_proved": False,
    "candidate_couplings": ["c1", "c2", "c3", "c4"],
    "candidate_couplings_fixed_by_GVH": False,
    "candidate_family_ready_for_static_weak_field": candidate_family_ready,
    "unique_GVH_prediction_ready": unique_GVH_prediction_ready,
    "observational_data_used": False,
    "next_step": "0.3.2.4 static spherical weak-field reduction for the candidate family",
}

if Path("/content").exists():
    EXPORT_DIR = Path("/content/gvh_exports")
else:
    EXPORT_DIR = Path.cwd() / "gvh_exports"

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

artifact_path = EXPORT_DIR / "gvh_0.3.2.3_explicit_field_equations_tensor_audit.json"
artifact_path.write_text(
    json.dumps(artifact, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

assert artifact_path.exists()
assert artifact_path.stat().st_size > 0
assert artifact["observational_data_used"] is False
assert artifact["unique_GVH_prediction_ready"] is False

print("PASS — tensor audit artifact created.")
print("Artifact path:", artifact_path)
print("Final status:", FINAL_STATUS)


PASS — tensor audit artifact created.
Artifact path: /content/gvh_exports/gvh_0.3.2.3_explicit_field_equations_tensor_audit.json
Final status: PASS-EXPLICIT-EOM-CANDIDATE-FAMILY_READY-FOR-WEAK-FIELD_BLOCKED-UNIQUE-PREDICTION



# Conclusion

0.3.2.3 advances the candidate program from a symbolic action to an explicit field-equation/tensor framework.

It establishes, conditionally on the 0.3.2.2 ansatz:

\[
u^\mu u_\mu=-1,
\]

an explicit candidate vector equation,

\[
\nabla_\alpha J^\alpha{}_\mu
+
c_4 a_\alpha\nabla_\mu u^\alpha
+
\lambda u_\mu=0,
\]

and the metric equation

\[
G_{\mu\nu}
=
8\pi G T_{\mu\nu}^{(m)}
+
T_{\mu\nu}^{(u)}.
\]

The displayed explicit form of \(T_{\mu\nu}^{(u)}\) is retained as `FORM_TO_VERIFY`, because a full independent arbitrary-coordinate tensor variation has not yet been reproduced here.

The basic symmetry, flat-background, normalization, and GR-recovery controls pass structurally. However,

\[
c_1,c_2,c_3,c_4
\]

remain free and are not derived by GVH.

Therefore the expected scientific status is:

```text
PASS-EXPLICIT-EOM-CANDIDATE-FAMILY_READY-FOR-WEAK-FIELD_BLOCKED-UNIQUE-PREDICTION
```

The next notebook may now perform a **static spherical weak-field reduction of the candidate family**, while keeping the \(c_i\) explicit and forbidding any claim of a unique GVH prediction until they are theoretically fixed.
